In [ ]:
import json
import os
import sys
import time
import traceback

REPO_ROOT = "/home/xilinx/Audio-Lab-PYNQ"
for path in (REPO_ROOT, os.path.join(REPO_ROOT, "GUI")):
    if path not in sys.path:
        sys.path.insert(0, path)

import ipywidgets as widgets
from IPython.display import display

from audio_lab_pynq import AudioLabOverlay
from audio_lab_pynq.hdmi_backend import AudioLabHdmiBackend
from audio_lab_pynq.hdmi_effect_state_mirror import (
    HdmiEffectStateMirror,
    PEDAL_MODEL_LABELS,
    AMP_MODEL_LABELS,
    CAB_MODEL_LABELS,
    PEDAL_MODELS,
    AMP_MODELS,
    CAB_MODELS,
    STATIC_PL_UTILIZATION,
)
from pynq_multi_fx_gui import (
    AppState,
    THEMES,
    make_pynq_static_render_cache,
    render_frame_800x480,
)

VARIANT = "compact-v2"
THEME = "pipboy-green" if "pipboy-green" in THEMES else None
PLACEMENT = "manual"
OFFSET_X = 0
OFFSET_Y = 0
RENDER_ON_CHANGE = True
THROTTLE_SECONDS = 0.20
RETURN_TO_SAFE_BYPASS_ON_START = True
RETURN_TO_SAFE_BYPASS_ON_EXIT = False
SHOW_RESOURCE_MONITOR = True

PRESET_NAMES = [
    "Safe Bypass", "Basic Clean", "Clean Sustain", "Light Crunch",
    "TS Lead", "RAT Rhythm", "Metal Tight", "Ambient Clean",
    "Solo Boost", "Noise Controlled High Gain",
    "DS-1 Crunch", "Big Muff Sustain", "Vintage Fuzz",
]

# pedal models: clean_boost, tube_screamer, rat, ds1, big_muff, fuzz_face, metal
# amp models: jc_clean, clean_combo, british_crunch, high_gain_stack
# cab models: 1x12, 2x12, 4x12
PEDAL_OPTIONS = [(PEDAL_MODEL_LABELS[name], name) for name in PEDAL_MODELS]
AMP_OPTIONS = [(AMP_MODEL_LABELS[name], name) for name in AMP_MODELS]
CAB_OPTIONS = [(CAB_MODEL_LABELS[name], name) for name in CAB_MODELS]
CATEGORY_OPTIONS = [
    ("PEDAL", "PEDAL"),
    ("AMP", "AMP"),
    ("CAB", "CAB"),
    ("REVERB", "REVERB"),
    ("EQ", "EQ"),
    ("COMPRESSOR", "COMPRESSOR"),
    ("NOISE SUPPRESSOR", "NOISE SUPPRESSOR"),
]

print("Loading AudioLabOverlay() once")
_overlay = AudioLabOverlay()
print("Creating AudioLabHdmiBackend once")
_backend = AudioLabHdmiBackend(_overlay)
_state = AppState()
_cache = make_pynq_static_render_cache()
mirror = HdmiEffectStateMirror(
    overlay=_overlay,
    hdmi_backend=_backend,
    app_state=_state,
    renderer=render_frame_800x480,
    render_cache=_cache,
    theme=THEME,
    variant=VARIANT,
    placement=PLACEMENT,
    offset_x=OFFSET_X,
    offset_y=OFFSET_Y,
)

_throttle_state = {"last_t": 0.0}
_last_operation = {"text": "init"}

def _throttled():
    now = time.time()
    if now - _throttle_state["last_t"] < THROTTLE_SECONDS:
        return True
    _throttle_state["last_t"] = now
    return False


def _safe_call(label, fn):
    _last_operation["text"] = label
    try:
        fn()
    except Exception as exc:
        traceback.print_exc()
        _status_label.value = "error: {}".format(exc)
        return False
    _refresh_status()
    _refresh_resource_panel()
    return True


# ---- Section A: Global / Preset -----------------------------------------
_safe_bypass_btn = widgets.Button(description="Safe Bypass",
                                   button_style="warning")
_basic_clean_btn = widgets.Button(description="Basic Clean")
_preset_dd = widgets.Dropdown(options=PRESET_NAMES, value="Basic Clean",
                              description="Preset:")
_apply_preset_btn = widgets.Button(description="Apply Preset",
                                    button_style="primary")
_render_btn = widgets.Button(description="Render")
_summary_btn = widgets.Button(description="Summary")
_status_label = widgets.HTML(value="<i>ready</i>")

_safe_bypass_btn.on_click(lambda b: _safe_call("safe_bypass",
                                               mirror.safe_bypass))
_basic_clean_btn.on_click(lambda b: _safe_call(
    "apply_chain_preset:Basic Clean",
    lambda: mirror.apply_chain_preset("Basic Clean")))
_apply_preset_btn.on_click(lambda b: _safe_call(
    "apply_chain_preset:" + str(_preset_dd.value),
    lambda: mirror.apply_chain_preset(_preset_dd.value)))
_render_btn.on_click(lambda b: _safe_call(
    "render", lambda: mirror.render(reason="manual")))

def _on_summary_click(_btn):
    _last_operation["text"] = "summary"
    try:
        data = mirror.summary()
        print(json.dumps(data, indent=2, sort_keys=True, default=str))
    except Exception:
        traceback.print_exc()
    _refresh_status()
    _refresh_resource_panel()
_summary_btn.on_click(_on_summary_click)

_section_a = widgets.VBox([
    widgets.HTML("<b>A. Global / Preset</b>"),
    widgets.HBox([_safe_bypass_btn, _basic_clean_btn, _render_btn, _summary_btn]),
    widgets.HBox([_preset_dd, _apply_preset_btn]),
    _status_label,
])

# ---- Section B: Selected FX + Model Dropdown ----------------------------
_category_dd = widgets.Dropdown(options=CATEGORY_OPTIONS, value="PEDAL",
                                description="Effect:")
_model_dd = widgets.Dropdown(options=PEDAL_OPTIONS,
                              value="tube_screamer",
                              description="Model:")
_apply_model_btn = widgets.Button(description="Apply Selected Model",
                                   button_style="primary")

def _refresh_model_options(change=None):
    cat = _category_dd.value
    if cat == "PEDAL":
        _model_dd.options = PEDAL_OPTIONS
        _model_dd.value = mirror.current_pedal_model
    elif cat == "AMP":
        _model_dd.options = AMP_OPTIONS
        _model_dd.value = mirror.current_amp_model
    elif cat == "CAB":
        _model_dd.options = CAB_OPTIONS
        _model_dd.value = mirror.current_cab_model
    elif cat == "REVERB":
        _model_dd.options = [("REVERB (no model)", "reverb")]
        _model_dd.value = "reverb"
    elif cat == "EQ":
        _model_dd.options = [("EQ (no model)", "eq")]
        _model_dd.value = "eq"
    elif cat == "COMPRESSOR":
        _model_dd.options = [("COMPRESSOR (no model)", "compressor")]
        _model_dd.value = "compressor"
    elif cat == "NOISE SUPPRESSOR":
        _model_dd.options = [("NOISE SUP (no model)", "noise_suppressor")]
        _model_dd.value = "noise_suppressor"

_category_dd.observe(_refresh_model_options, names="value")

def _apply_selected_model():
    cat = _category_dd.value
    model = _model_dd.value
    if cat == "PEDAL":
        mirror.set_pedal_model(model,
                                drive=int(_pedal_drive.value),
                                tone=int(_pedal_tone.value),
                                level=int(_pedal_level.value),
                                mix=int(_pedal_mix.value),
                                enabled=bool(_pedal_on.value))
    elif cat == "AMP":
        mirror.set_amp_model(model,
                              gain=int(_amp_gain.value),
                              bass=int(_amp_bass.value),
                              mid=int(_amp_mid.value),
                              treble=int(_amp_treble.value),
                              presence=int(_amp_presence.value),
                              resonance=int(_amp_resonance.value),
                              master=int(_amp_master.value))
    elif cat == "CAB":
        mirror.set_cab_model(model,
                              air=int(_cab_air.value),
                              enabled=bool(_cab_on.value))
    elif cat == "REVERB":
        mirror.reverb(enabled=bool(_reverb_on.value),
                       mix=int(_reverb_mix.value),
                       decay=int(_reverb_decay.value),
                       tone=int(_reverb_tone.value))
    elif cat == "EQ":
        mirror.eq(enabled=True, low=int(_eq_low.value),
                   mid=int(_eq_mid.value), high=int(_eq_high.value))
    elif cat == "COMPRESSOR":
        mirror.set_compressor_settings(enabled=True, threshold=45,
                                        ratio=35, response=45, makeup=50)
    elif cat == "NOISE SUPPRESSOR":
        mirror.set_noise_suppressor_settings(enabled=True, threshold=25,
                                              decay=84, damp=85)

_apply_model_btn.on_click(lambda b: _safe_call(
    "apply_selected_model:{}/{}".format(_category_dd.value, _model_dd.value),
    _apply_selected_model))

def _on_model_change(change):
    if not RENDER_ON_CHANGE or _throttled():
        return
    cat = _category_dd.value
    model = change["new"]
    if cat == "PEDAL":
        _safe_call("pedal_change:" + str(model),
                    lambda: mirror.set_pedal_model(model))
    elif cat == "AMP":
        _safe_call("amp_change:" + str(model),
                    lambda: mirror.set_amp_model(model))
    elif cat == "CAB":
        _safe_call("cab_change:" + str(model),
                    lambda: mirror.set_cab_model(model))

_model_dd.observe(_on_model_change, names="value")

_section_b = widgets.VBox([
    widgets.HTML("<b>B. Selected FX / Model Dropdown</b>"),
    widgets.HBox([_category_dd, _model_dd, _apply_model_btn]),
])

# ---- Section C: Pedal controls ------------------------------------------
_pedal_on = widgets.Checkbox(value=True, description="PEDAL ON")
_pedal_drive = widgets.IntSlider(value=45, min=0, max=100, step=1, description="drive")
_pedal_tone = widgets.IntSlider(value=55, min=0, max=100, step=1, description="tone")
_pedal_level = widgets.IntSlider(value=65, min=0, max=100, step=1, description="level")
_pedal_mix = widgets.IntSlider(value=100, min=0, max=100, step=1, description="mix")
_pedal_apply_btn = widgets.Button(description="Apply Pedal")

def _apply_pedal():
    mirror.set_pedal_model(_model_dd.value if _category_dd.value == "PEDAL"
                            else mirror.current_pedal_model,
                            drive=int(_pedal_drive.value),
                            tone=int(_pedal_tone.value),
                            level=int(_pedal_level.value),
                            mix=int(_pedal_mix.value),
                            enabled=bool(_pedal_on.value))
_pedal_apply_btn.on_click(lambda b: _safe_call("apply_pedal", _apply_pedal))
_section_c = widgets.VBox([
    widgets.HTML("<b>C. Pedal controls</b>"),
    widgets.HBox([_pedal_on, _pedal_apply_btn]),
    _pedal_drive, _pedal_tone, _pedal_level, _pedal_mix,
])

# ---- Section D: Amp controls --------------------------------------------
_amp_on = widgets.Checkbox(value=True, description="AMP ON")
_amp_gain = widgets.IntSlider(value=60, min=0, max=100, step=1, description="gain")
_amp_bass = widgets.IntSlider(value=55, min=0, max=100, step=1, description="bass")
_amp_mid = widgets.IntSlider(value=60, min=0, max=100, step=1, description="middle")
_amp_treble = widgets.IntSlider(value=55, min=0, max=100, step=1, description="treble")
_amp_presence = widgets.IntSlider(value=45, min=0, max=100, step=1, description="presence")
_amp_resonance = widgets.IntSlider(value=35, min=0, max=100, step=1, description="resonance")
_amp_master = widgets.IntSlider(value=70, min=0, max=100, step=1, description="master")
_amp_apply_btn = widgets.Button(description="Apply Amp")
def _apply_amp():
    mirror.set_amp_model(_model_dd.value if _category_dd.value == "AMP"
                          else mirror.current_amp_model,
                          gain=int(_amp_gain.value),
                          bass=int(_amp_bass.value),
                          mid=int(_amp_mid.value),
                          treble=int(_amp_treble.value),
                          presence=int(_amp_presence.value),
                          resonance=int(_amp_resonance.value),
                          master=int(_amp_master.value))
_amp_apply_btn.on_click(lambda b: _safe_call("apply_amp", _apply_amp))
_section_d = widgets.VBox([
    widgets.HTML("<b>D. Amp controls (GAIN/BASS/MIDDLE/TREBLE/PRESENCE/RESONANCE/MASTER/CHARACTER)</b>"),
    widgets.HBox([_amp_on, _amp_apply_btn]),
    _amp_gain, _amp_bass, _amp_mid, _amp_treble,
    _amp_presence, _amp_resonance, _amp_master,
])

# ---- Section E: Cab controls --------------------------------------------
_cab_on = widgets.Checkbox(value=True, description="CAB ON")
_cab_model_dd = widgets.Dropdown(options=CAB_OPTIONS,
                                  value="4x12", description="cab")
_cab_air = widgets.IntSlider(value=40, min=0, max=100, step=1, description="air")
_cab_apply_btn = widgets.Button(description="Apply Cab")
def _apply_cab():
    mirror.set_cab_model(_cab_model_dd.value,
                          air=int(_cab_air.value),
                          enabled=bool(_cab_on.value))
_cab_apply_btn.on_click(lambda b: _safe_call("apply_cab", _apply_cab))
_section_e = widgets.VBox([
    widgets.HTML("<b>E. Cab controls</b>"),
    widgets.HBox([_cab_on, _cab_model_dd, _cab_apply_btn]),
    _cab_air,
])

# ---- Section F: Reverb / EQ ---------------------------------------------
_reverb_on = widgets.Checkbox(value=True, description="REVERB ON")
_reverb_mix = widgets.IntSlider(value=25, min=0, max=100, step=1, description="rvb mix")
_reverb_decay = widgets.IntSlider(value=50, min=0, max=100, step=1, description="rvb decay")
_reverb_tone = widgets.IntSlider(value=65, min=0, max=100, step=1, description="rvb tone")
_reverb_apply_btn = widgets.Button(description="Apply Reverb")
def _apply_reverb():
    mirror.reverb(enabled=bool(_reverb_on.value),
                   mix=int(_reverb_mix.value),
                   decay=int(_reverb_decay.value),
                   tone=int(_reverb_tone.value))
_reverb_apply_btn.on_click(lambda b: _safe_call("apply_reverb", _apply_reverb))

_eq_low = widgets.IntSlider(value=50, min=0, max=100, step=1, description="eq low")
_eq_mid = widgets.IntSlider(value=55, min=0, max=100, step=1, description="eq mid")
_eq_high = widgets.IntSlider(value=60, min=0, max=100, step=1, description="eq high")
_eq_apply_btn = widgets.Button(description="Apply EQ")
def _apply_eq():
    mirror.eq(enabled=True, low=int(_eq_low.value),
               mid=int(_eq_mid.value), high=int(_eq_high.value))
_eq_apply_btn.on_click(lambda b: _safe_call("apply_eq", _apply_eq))
_section_f = widgets.VBox([
    widgets.HTML("<b>F. Reverb / EQ</b>"),
    widgets.HBox([_reverb_on, _reverb_apply_btn]),
    _reverb_mix, _reverb_decay, _reverb_tone,
    widgets.HBox([_eq_apply_btn]),
    _eq_low, _eq_mid, _eq_high,
])

# ---- Section G: Resource Monitor ---------------------------------------
_resource_html = widgets.HTML(value="<i>resource monitor pending…</i>")
_section_g = widgets.VBox([
    widgets.HTML("<b>G. Resource Monitor (PS / GUI / HDMI)</b>"),
    _resource_html,
])

def _fmt_pct(value):
    if value is None:
        return "n/a"
    return "{:.1f}%".format(float(value))

def _fmt_mb(kb):
    if kb is None:
        return "n/a"
    return "{:.1f} MB".format(float(kb) / 1024.0)

def _fmt_sec(value):
    if value is None:
        return "n/a"
    try:
        return "{:.3f} s".format(float(value))
    except (TypeError, ValueError):
        return str(value)

def _refresh_resource_panel():
    if not SHOW_RESOURCE_MONITOR:
        return
    data = mirror.resource_summary()
    bits = data.get("vdma_error_bits") or {}
    bits_text = ", ".join(
        "{}={}".format(k, v) for k, v in sorted(bits.items()))
    pl = data.get("pl_utilization") or STATIC_PL_UTILIZATION
    lwr = data.get("last_frame_write") or {}
    rows = [
        ("selected FX", data.get("selected_fx")),
        ("category", data.get("selected_model_category")),
        ("dropdown", "{} ({})".format(data.get("dropdown_short_label"),
                                       data.get("dropdown_label"))),
        ("pedal model", data.get("current_pedal_label")),
        ("amp model", data.get("current_amp_label")),
        ("cab model", data.get("current_cab_label")),
        ("last edited", data.get("last_edited_effect")),
        ("last operation", _last_operation["text"]),
        ("render_s", _fmt_sec(data.get("render_s"))),
        ("compose_s", _fmt_sec(data.get("compose_s"))),
        ("copy_s", _fmt_sec(data.get("framebuffer_copy_s"))),
        ("total update", _fmt_sec(data.get("total_update_s"))),
        ("VDMA DMASR", data.get("vdma_dmasr")),
        ("VDMA error bits", bits_text),
        ("VTC ctl", data.get("vtc_ctl")),
        ("placement", lwr.get("placement")),
        ("offset_x", lwr.get("offset_x")),
        ("offset_y", lwr.get("offset_y")),
        ("proc CPU", _fmt_pct(data.get("proc_cpu_pct"))),
        ("sys CPU", _fmt_pct(data.get("sys_cpu_pct"))),
        ("proc RSS", _fmt_mb(data.get("proc_rss_kb"))),
        ("Mem available", _fmt_mb(data.get("mem_avail_kb"))),
        ("Mem total", _fmt_mb(data.get("mem_total_kb"))),
        ("temperature", "{} C".format(data.get("temperature_c"))
                          if data.get("temperature_c") is not None else "n/a"),
        ("PL LUT (static)", pl.get("lut")),
        ("PL Reg (static)", pl.get("registers")),
        ("PL BRAM (static)", pl.get("bram_36k")),
        ("PL DSP (static)", pl.get("dsp48")),
    ]
    rows_html = "".join(
        "<tr><td style='padding:1px 8px'>{}</td>"
        "<td style='padding:1px 8px'><code>{}</code></td></tr>".format(k, v)
        for k, v in rows)
    _resource_html.value = (
        "<table style='border-collapse:collapse'>"
        "<thead><tr><th align='left'>field</th>"
        "<th align='left'>value</th></tr></thead>"
        "<tbody>{}</tbody></table>".format(rows_html))

def _refresh_status():
    actual = mirror.get_selected_fx_actual()
    dd = getattr(mirror.app_state, "dropdown_short_label", "")
    _status_label.value = (
        "<code>SELECTED FX = {}</code> &nbsp; <code>[{} ▼]</code>"
        .format(actual, dd))

# ---- Initial state -------------------------------------------------------
if RETURN_TO_SAFE_BYPASS_ON_START:
    print("Returning to safe bypass on start")
    mirror.safe_bypass()
else:
    mirror.render(reason="initial")
_refresh_model_options()
_refresh_status()
_refresh_resource_panel()

ui = widgets.VBox([
    _section_a, _section_b, _section_c, _section_d, _section_e,
    _section_f, _section_g,
])
display(ui)
print("Notebook ready. Operate widgets above; HDMI GUI mirrors each edit via"
      " HdmiEffectStateMirror -> AudioLabOverlay -> 800x480 x=0,y=0 render.")
